# Time-series analysis & change detection jupyter notebook using radar and optical satellite data.


In [ ]:
from IPython.display import HTML
import uuid
def Younes_clock(): 
    unique_id = str(uuid.uuid4()).replace('-', '') 
    text = f""" <span style="font-weight:bold;font-size:1.2em;display:flex;align-items:center;gap:8px;"> <img src="https://media.licdn.com/dms/image/v2/C4D03AQG1blt50-okxQ/profile-displayphoto-shrink_400_400/profile-displayphoto-shrink_400_400/0/1661855961507?e=1753920000&v=beta&t=dM1-FiugifL0hzVUeSetkUVH0RL9JYV7KqAhdtOBgCA" alt="Dr. Younes Abid" style="width:32px;height:32px;border-radius:50%;vertical-align:middle;"> <a href="https://www.linkedin.com/in/younes-abid/" target="_blank" style="color:inherit;text-decoration:underline;"> Dr. Younes Abid </a> | Arab youth hackathon 2026 | <span id="date_{unique_id}"></span> | Time: <span id="time_{unique_id}"></span> </span> <script> function updateClock_{unique_id}(){{ var now = new Date(); var dateStr = now.getFullYear() + '-' + String(now.getMonth()+1).padStart(2, '0') + '-' + String(now.getDate()).padStart(2, '0'); var timeStr = now.toLocaleTimeString(); var dateElem = document.getElementById('date_{unique_id}'); var timeElem = document.getElementById('time_{unique_id}'); if(dateElem && timeElem){{ dateElem.innerHTML = dateStr; timeElem.innerHTML = timeStr; }} }} setInterval(updateClock_{unique_id}, 1000); updateClock_{unique_id}(); </script> """ 
    
    return HTML(text)
Younes_clock()
S1: Time-series Analysis & Change Detection
Missile Damage Detection in the June 2025 Iran-Israel Conflict (Tel Aviv, Fordow, Tehran)
Earth Observation with Sentinel-1/2 & GEE Python API

Table of Contents
Introduction to Earth Observation (EO) & Change Detection
Case Context: June 2025 Iran-Israel Missile Strikes
Downloading Sentinel-1 and Sentinel-2 Data Using Copernicus Data Space Browser
Load Data, Coregistration, and Preprocessing
Constructing and Visualizing Time Series
Patch-Based NDBI Analysis
Change Detection Algorithms
Patched Change Detection


! pip install folium rasterio xarray pyproj shapely rioxarray matplotlib
1. Introduction: EO Time Series & Conflict Monitoring
Theory
Earth Observation (EO) time series analysis involves monitoring changes in the Earth's surface over time using satellite imagery. By analyzing sequences of images, we can detect trends, abrupt changes, or anomalies caused by natural or human-induced events.

Change detection is especially critical for conflict monitoring, as it allows us to rapidly identify and assess damage from events such as missile strikes, explosions, or other military activities. By comparing satellite data before and after such events, we can quantify the impact and support humanitarian or strategic responses.

Practice
Let's visualize a simple NDBI (Normalized Difference Built-up Index) time series for a conflict-affected Area of Interest (AOI). NDBI is a common index derived from satellite data (e.g., Sentinel-2) that indicates built-up area and can reveal abrupt changes due to development.
NDBI = (SWIR - NIR) / (SWIR + NIR)



In [ ]:
# Example: Simulated NDBI time series for a conflict AOI

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Simulate dates (e.g., June 2025)
dates = pd.date_range("2025-06-01", "2025-06-30", freq="D")
# Simulate NDBI values with a sudden decrease (e.g., bombing on June 21)
ndbi = np.ones(len(dates)) * 0.4
ndbi[21:] -= 0.2  # Sudden decrease after June 21

plt.figure(figsize=(10,4))
plt.plot(dates, ndbi, marker='o')
plt.axvline(pd.Timestamp("2025-06-22"), color='red', linestyle='--', label='bombing event')
plt.title("Simulated NDBI Time Series for Conflict AOI (June 2025)")
plt.xlabel("Date")
plt.ylabel("NDBI")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
Exercise
Look at the NDBI time series plot above.
Question: On which date does the most significant change occur?
Hint: Identify the date where the NDBI value increases sharply.

FAQ: NDBI and Urban Damage Detection
Q1: Does NDBI increase or decrease after bombing or missile strikes?

A: NDBI usually decreases after heavy urban damage. When buildings collapse, the area becomes less "built-up" and more like exposed ground or debris, which typically has a lower NDBI value. Dust and rubble further reduce the NDBI signal.
Q2: Are the simulated NDBI values in our example realistic?

A: The simulated drop (e.g., from 0.4 to 0.2) is reasonable for demonstration, but real-world changes may be smaller or larger depending on the severity and type of destruction. Actual NDBI changes are often subtle and require careful analysis.
Q3: Does our example show the curve for one pixel, an AOI, or the entire image?

A: The example simulates the NDBI time series for a single AOI or pixel. In practice, it is common to plot the mean or median NDBI over a small AOI (such as the impact site) for more robust results.
Summary:

After bombing, expect NDBI to decrease due to destruction and debris.
The simulation is illustrative; real changes may vary.
The plot represents a single AOI or pixel, not the whole image.
2. Case Context: June 2025 Iran-Israel Missile Strikes
Theory
In late June 2025, a series of missile strikes occurred between Iran and Israel, targeting key urban and strategic sites such as Tel Aviv (Israel), Fordow nuclear facility (Iran), and areas in Tehran. These events resulted in significant infrastructure damage, fires, and debris.
Earth Observation (EO) satellites, especially Sentinel-1 (SAR) and Sentinel-2 (optical), provide valuable data to monitor and assess the impact of such conflicts.
EO signatures of missile damage include:

Sudden changes in built-up indices (e.g., NDBI)
Changes in SAR backscatter (due to debris, rubble, or surface roughness)
Appearance of burn scars or thermal anomalies
Visible craters or collapsed structures in high-resolution imagery
Practice
Let's map the Areas of Interest (AOIs) for this case study:

Tel Aviv, Israel (urban target)
Fordow, Iran (nuclear facility)
Tehran, Iran (urban/industrial target)


# Visualize AOIs (Tel Aviv, Fordow, Tehran) as polygons on an interactive map using folium

import folium

# Define AOI center coordinates (latitude, longitude)
aoi_centers = {
    "Tel Aviv, Israel": [32.0853, 34.7818],
    "Fordow, Iran": [34.8845, 50.9981],
    "Tehran, Iran": [35.6892, 51.3890]
}

# Define AOI polygons (simple rectangles for demonstration)
aoi_polygons = {
    "Tel Aviv, Israel": [
        [32.10, 34.76],
        [32.10, 34.80],
        [32.07, 34.80],
        [32.07, 34.76],
        [32.10, 34.76]
    ],
    "Fordow, Iran": [
        [34.89, 51.01],
        [34.89, 50.98],
        [34.88, 50.98],
        [34.88, 51.01],
        [34.89, 51.01]
    ],
    "Tehran, Iran": [
        [35.70, 51.40],
        [35.70, 51.38],
        [35.68, 51.38],
        [35.68, 51.40],
        [35.70, 51.40]
    ]
}

# Create a folium map centered between the AOIs
m = folium.Map(location=[34.5, 44.0], zoom_start=5)

# Add markers and polygons for each AOI
for name, center in aoi_centers.items():
    folium.Marker(
        location=center,
        popup=name,
        icon=folium.Icon(color='red', icon='info-sign')
    ).add_to(m)
    folium.Polygon(
        locations=aoi_polygons[name],
        color='blue',
        fill=True,
        fill_opacity=0.2,
        popup=f"{name} AOI"
    ).add_to(m)

# Display the map
m
Exercise
Question:
List two EO indicators that can help detect missile damage in urban or strategic sites using satellite imagery.

Hint: Think about what changes you would expect to see in SAR or optical data after a strike.

FAQ: EO Damage Detection in Conflict Zones
Q1: Why are both SAR and optical data used for conflict damage assessment?

A: SAR (e.g., Sentinel-1) can detect structural changes and debris regardless of weather or daylight, while optical data (e.g., Sentinel-2) provides detailed visual and spectral information. Combining both increases detection reliability.
Q2: What are typical EO signatures of missile damage?

A: Sudden decrease in NDBI, abrupt change in SAR backscatter, appearance of burn scars, and visible destruction in high-resolution images.
Q3: Can EO detect all types of damage?

A: No. Some damage may be too small to detect at Sentinel resolution, or may be obscured by clouds, smoke, or rapid cleanup.
3. Downloading Sentinel-1 and Sentinel-2 Data Using Copernicus Data Space Browser
Theory
The Copernicus Data Space Ecosystem provides an intuitive browser interface for accessing and downloading Sentinel-1 and Sentinel-2 data. This method is ideal for users who prefer a graphical interface over programmatic solutions.

Practice
Follow these steps to download Sentinel-1 and Sentinel-2 data using the Copernicus Data Space Browser:

Open the Copernicus Data Space Browser
Navigate to the following URL:
Copernicus Data Space Browser

Set Your Area of Interest (AOI)

Use the map interface to zoom in and pan to your desired location (e.g., Tel Aviv, Fordow, Tehran).
Draw a polygon or rectangle to define your AOI using the tools provided.
Select the Dataset

Choose the dataset you want to download:
Sentinel-2 Level-2A for optical data.
Sentinel-1 for SAR data.
Adjust the cloud coverage filter (e.g., set it to 30% or less for Sentinel-2).
Set the Date Range

Use the calendar tool to select the date range for your query (e.g., June 2025).
You can choose a single date or a range depending on your analysis needs.
Search for Data

Click the search button to retrieve available data products for your AOI and date range.
Review the results displayed on the map and in the product list.
Preview and Download

Click on a product to view its metadata and preview the image.
If the product meets your requirements, click the Download button to save it to your computer.
Optional: Use Filters

Apply additional filters such as cloud coverage, resolution, or acquisition mode to refine your search.
Exercise
Define an AOI for a conflict-affected region (e.g., Tel Aviv, Fordow, Tehran).
Set the date range to June 2025 and filter Sentinel-2 data with cloud coverage below 30%.
Download one Sentinel-2 and one Sentinel-1 product for further analysis.
FAQ
Q1: Do I need an account to download data?

Yes, you need to create a free account on the Copernicus Data Space Ecosystem to download data.
Q2: Can I download multiple products at once?

Yes, you can select multiple products and download them as a batch.
Q3: What format are the downloaded files?

Sentinel-2 data is typically provided in .SAFE format, while Sentinel-1 data is provided in .zip format.
Q4: Can I use the downloaded data in Python?

Yes, you can process the data using libraries like rasterio, geemap, or numpy.
4. Load Data, Coregistration, and Preprocessing
Theory
After downloading Sentinel-1 and Sentinel-2 data, the next step is to load the data, perform coregistration, and preprocess it for analysis. Coregistration ensures that images from different dates or sensors align spatially, which is critical for accurate change detection.

Practice


In [ ]:
# Define AOI polygons (simple rectangles for demonstration)
aoi_polygons = {
    "Tel Aviv, Israel": [
        [32.10, 34.76],
        [32.10, 34.80],
        [32.07, 34.80],
        [32.07, 34.76],
        [32.10, 34.76]
    ],
    "Fordow, Iran": [
        [34.89, 51.01],
        [34.89, 50.98],
        [34.88, 50.98],
        [34.88, 51.01],
        [34.89, 51.01]
    ],
    "Tehran, Iran": [
        [35.70, 51.40],
        [35.70, 51.38],
        [35.68, 51.38],
        [35.68, 51.40],
        [35.70, 51.40]
    ]
}


import os
import xml.etree.ElementTree as ET
import glob
import rasterio
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
import numpy as np
import xarray as xr
import json
from pyproj import Transformer
from shapely.geometry import Polygon, mapping

def list_data(data_dir):
    """List all .SAFE.zip files in the data directory."""
    files = glob.glob(os.path.join(data_dir, "*.SAFE.zip"))
    print("Downloaded data:")
    for file in files:
        print(file)
    return files

def unzip_data(files, output_dir):
    """Unzip all .SAFE.zip files."""
    for file in files:
        os.system(f"unzip {file} -d {output_dir}")
    print("Unzipping completed.")
    
def extract_sentinel2_calibration(unzipped_dir, band_suffixes):
    """
    Extract calibration information (gain, offset, and BOA quantification value) 
    for Sentinel-2 products and organize them into a dictionary.

    Parameters:
        unzipped_dir (str): Directory containing unzipped .SAFE files.
        band_suffixes (list): List of band suffixes to search for (e.g., ["B11_20m.jp2", "B08_10m.jp2"]).

    Returns:
        dict: A dictionary containing calibration information for each Sentinel-2 product.
    """
    calibration_data = {}

    for root, dirs, files in os.walk(unzipped_dir):
        # Look for metadata files
        for file in files:
            if file.endswith("MTD_MSIL2A.xml"):  # Sentinel-2 metadata file
                metadata_path = os.path.join(root, file)
                product_name = os.path.basename(root)
                print(f"Processing metadata file: {metadata_path} for product: {product_name}")
                
                # Parse the XML metadata file
                tree = ET.parse(metadata_path)
                root = tree.getroot()
                
                # Namespace handling
                ns = {'n1': 'https://psd-15.sentinel2.eo.esa.int/PSD/User_Product_Level-2A.xsd'}
                
                # Extract BOA_QUANTIFICATION_VALUE
                quantification_path = ".//*BOA_QUANTIFICATION_VALUE"
                quantification_element = root.find(quantification_path, ns)
                quantification_value = float(quantification_element.text) if quantification_element is not None else None
                
                # Initialize product entry in the dictionary
                calibration_data[product_name] = {}

                # Extract gain and offset values for each band
                for suffix in band_suffixes:
                    band_name = suffix.split("_")[0][1:]  # Extract band name (e.g., "11")
                    band_name = f"{int(band_name)}"
                    
                    # Find gain
                    gain_path = f".//*PHYSICAL_GAINS[@bandId='{band_name}']"
                    gain_element = root.find(gain_path, ns)
                    gain = float(gain_element.text) if gain_element is not None else None
                    
                    # Find offset
                    offset_path = f".//*BOA_ADD_OFFSET[@band_id='{band_name}']"
                    offset_element = root.find(offset_path, ns)
                    offset = float(offset_element.text) if offset_element is not None else None
                    
                    # Add band calibration info to the product entry
                    calibration_data[product_name][suffix.split("_")[0]] = {
                        "gain": gain,
                        "offset": offset,
                        "boa_quantification": quantification_value
                    }

    return calibration_data

def select_bands(unzipped_dir, band_suffixes, target_crs="EPSG:4326", resample=10, coregister=True, AOI=None, output_dir="./processed_data", calibrate=False):
    """
    Select, crop, reproject, resample, and optionally coregister bands from Sentinel-2 .SAFE files.
    
    Parameters:
        unzipped_dir (str): Directory containing unzipped .SAFE files
        band_suffixes (list): List of band suffixes to select (e.g., ["B11_20m.jp2", "B08_10m.jp2"])
        target_crs (str): CRS to reproject bands to (default: "EPSG:4326")
        resample (int): Resolution to resample bands to (default: 10 meters)
        coregister (bool): Whether to coregister bands (default: True)
        AOI (list): Area of Interest polygon in EPSG:4326 coordinates (default: None)
        output_dir (str): Directory to save processed bands (default: "./processed_data")
    """
    if calibrate:
        calibration_info = extract_sentinel2_calibration(unzipped_dir, band_suffixes)
        
    os.makedirs(output_dir, exist_ok=True)
    
    # Convert AOI to (lon, lat) format if provided
    if AOI:
        AOI = [(lon, lat) for lat, lon in AOI]  # Convert to (lon, lat) format
        aoi_polygon = Polygon(AOI)
    else:
        aoi_polygon = None
        
    for root, dirs, files in os.walk(unzipped_dir):
        if "IMG_DATA" not in root:
            continue
        safe_name = os.path.basename(os.path.dirname(os.path.dirname(root)))
        selected_bands = {}
        
        # Step 1: Select bands
        for file in files:
            for suffix in band_suffixes:
                if file.endswith(suffix):
                    band_path = os.path.join(root, file)
                    selected_bands[suffix] = band_path
                    break

        if not selected_bands:
            continue
            
        print(f"\nProcessing {safe_name}")
        processed_bands = {}
        
        # Step 2: Process each selected band
        for suffix, band_path in selected_bands.items():
            print(f"Processing {suffix}")
            
            with rasterio.open(band_path) as src:
                print(f"Original CRS: {src.crs}, Resolution: {src.res}")
                
                # Transform AOI to raster's CRS if provided
                if aoi_polygon:
                    transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
                    transformed_coords = [transformer.transform(x, y) for x, y in AOI]
                    transformed_polygon = Polygon(transformed_coords)
                    
                    # Check if AOI intersects with raster
                    raster_polygon = Polygon([
                        (src.bounds.left, src.bounds.bottom),
                        (src.bounds.right, src.bounds.bottom),
                        (src.bounds.right, src.bounds.top),
                        (src.bounds.left, src.bounds.top)
                    ])
                    
                    if not transformed_polygon.intersects(raster_polygon):
                        print(f"AOI doesn't intersect with {suffix}. Skipping...")
                        continue
                        
                    # Crop to AOI
                    try:
                        out_image, out_transform = mask(src, [mapping(transformed_polygon)], crop=True)
                    except ValueError as e:
                        print(f"Error cropping {suffix}: {e}")
                        continue
                else:
                    out_image = src.read()
                    out_transform = src.transform
                
                if calibrate:
                    product_name = [part for part in root.split(os.sep) if part.endswith(".SAFE")][0]
                    # Apply calibration if available
                    gain = calibration_info[product_name][suffix.split("_")[0]]["gain"]
                    offset = calibration_info[product_name][suffix.split("_")[0]]["offset"]
                    boa_quantification = calibration_info[product_name][suffix.split("_")[0]]["boa_quantification"]
                    
                    out_image = (out_image * gain + offset) / boa_quantification
                    print(f"Applied calibration: gain={gain}, offset={offset}, BOA quantification={boa_quantification}")
                    
                # Determine target resolution
                if target_crs == "EPSG:4326":
                    target_res_deg = resample / 111320  # Approximate conversion
                    dst_resolution = (target_res_deg, target_res_deg)
                else:
                    dst_resolution = (resample, resample)
                
                # Calculate new transform
                transform, width, height = calculate_default_transform(
                    src.crs, target_crs, 
                    src.width, src.height, 
                    *src.bounds, 
                    resolution=dst_resolution
                )
                
                # Update metadata
                meta = src.meta.copy()
                meta.update({
                    'driver': 'GTiff',
                    'crs': target_crs,
                    'transform': transform,
                    'width': width,
                    'height': height,
                    'count': out_image.shape[0],
                    'dtype': out_image.dtype
                })
                
                # Create output path
                output_path = os.path.join(
                    output_dir, 
                    f"{safe_name}_{suffix.replace('.jp2', '')}_processed.tif"
                )
                
                # Reproject and save
                with rasterio.open(output_path, 'w', **meta) as dst:
                    for i in range(out_image.shape[0]):
                        reproject(
                            source=out_image[i],
                            destination=rasterio.band(dst, i+1),
                            src_transform=out_transform,
                            src_crs=src.crs,
                            dst_transform=transform,
                            dst_crs=target_crs,
                            resampling=Resampling.bilinear
                        )
                
                processed_bands[suffix] = output_path
                print(f"Processed {suffix} saved to {output_path}")
                print("-"*100)
    
    print("\nProcessing complete!")
    
def resize_band(band, src_profile, target_shape):
    """
    Resize a raster band to match the target shape using bilinear resampling.
    """
    with rasterio.MemoryFile() as memfile:
        with memfile.open(
            driver="GTiff",
            height=band.shape[0],
            width=band.shape[1],
            count=1,
            dtype=band.dtype,
            crs=src_profile["crs"],
            transform=src_profile["transform"],
        ) as src:
            src.write(band, 1)
            resized_band = src.read(
                out_shape=(1, target_shape[0], target_shape[1]),
                resampling=Resampling.bilinear,
            )
    return resized_band[0]

def compute_ndbi(input_dir, output_dir):
    """
    Compute NDBI for each Sentinel-2 product in the input directory and save the result.
    """
    os.makedirs(output_dir, exist_ok=True)
    files = os.listdir(input_dir)
    
    # Group files by product name
    products = {}
    for file in files:
        if file.endswith("_processed.tif"):
            product_name = "_".join(file.split("_")[:4])
            products.setdefault(product_name, []).append(file)

    for product_name, band_files in products.items():
        try:
            print(f"\nProcessing {product_name}...")
            
            b08_file = next((f for f in band_files if "B08_10m" in f), None)
            b11_file = next((f for f in band_files if "B11_20m" in f), None)

            if not b08_file or not b11_file:
                print(f"Skipping {product_name}: Missing required bands.")
                continue

            with rasterio.open(os.path.join(input_dir, b08_file)) as b08_src:
                b08 = b08_src.read(1).astype(np.float32)
                b08_profile = b08_src.profile

            with rasterio.open(os.path.join(input_dir, b11_file)) as b11_src:
                b11 = b11_src.read(1).astype(np.float32)
                b11_profile = b11_src.profile

            print(f"B08 stats: min={np.min(b08)}, max={np.max(b08)}")
            print(f"B11 stats: min={np.min(b11)}, max={np.max(b11)}")

            if b08.shape != b11.shape:
                print(f"Resizing bands for {product_name}...")
                if b08.shape[0] > b11.shape[0] or b08.shape[1] > b11.shape[1]:
                    b11 = resize_band(b11, b11_profile, b08.shape)
                else:
                    b08 = resize_band(b08, b08_profile, b11.shape)

            denominator = b11 + b08
            epsilon = 1e-6
            
            invalid_mask = denominator == 0
            denominator[invalid_mask] += epsilon

            ndbi = (b11 - b08) / denominator

            ndbi = np.clip(ndbi, -1, 1)
            ndbi[np.isnan(ndbi)] = 0

            print(f"NDBI stats: min={np.min(ndbi)}, max={np.max(ndbi)}")

            b08_profile.update(dtype=rasterio.float32, count=1)

            output_path = os.path.join(output_dir, f"{product_name}_NDBI.tif")
            with rasterio.open(output_path, "w", **b08_profile) as dst:
                dst.write(ndbi, 1)

            print(f"NDBI saved for {product_name} at {output_path}")

        except Exception as e:
            print(f"Error processing {product_name}: {e}")


In [ ]:
# Define the directories
data_dir = "./data"
unzipped_dir = "./unzipped_data"
aoi_polygon = [
    [32.15, 34.70],  # NW corner
    [32.15, 34.85],  # NE corner
    [32.05, 34.85],  # SE corner
    [32.05, 34.70],  # SW corner
    [32.15, 34.70]   # Closing the polygon
]


files = list_data(data_dir)


In [ ]:
unzip_data(files, unzipped_dir)
Why Do We Apply Calibration?
Before computing indices such as NDBI, we often convert the raw pixel values stored in Sentinel-2 bands into physically meaningful surface reflectance values. This step is called radiometric calibration.

Acronyms
DN = Digital Number
BOA = Bottom Of Atmosphere
Calibration is important because:

Raw digital values are scaled numbers, not direct reflectance values.
Images from different dates should be converted to a comparable radiometric scale before change detection.
Index calculations such as NDBI become more reliable when the input bands represent corrected physical quantities instead of sensor-scaled integers.
Calibration Formula
For each band, the conversion used in this notebook is:

$$ \mathrm{BOA\ Reflectance} = \frac{(DN \times Gain) + Offset}{BOA\ Quantification\ Value} $$

What Each Parameter Means
DN (Digital Number): the original pixel value recorded in the band image.
Gain: a multiplicative factor used to scale the raw pixel value.
Offset: an additive correction applied after scaling.
BOA Quantification Value: a normalization factor used to convert the scaled values into Bottom-Of-Atmosphere reflectance.
BOA Reflectance: the final calibrated value representing the surface reflectance measured after atmospheric correction.
Why This Matters for Change Detection
If two images are compared without calibration, some of the observed differences may come from scaling and metadata conventions rather than real surface changes. By calibrating first, we reduce that risk and make temporal comparisons more defensible.

FAQ
Q1: How do we get BOA reflectance?

BOA reflectance is obtained after atmospheric correction, which removes atmospheric effects from the satellite signal.
Q2: How do they get the BOA quantification value?

It is provided by the Sentinel-2 product metadata. ESA defines this scaling factor during product generation, and we read it from the XML metadata file.
Q3: Is there another level besides BOA?

Yes. A common alternative is TOA reflectance.
Q4: What does TOA mean?

TOA = Top Of Atmosphere. It is the reflectance measured at the satellite before atmospheric correction.
Q5: Which one should we prefer for change detection?

BOA is usually preferred because it is closer to true surface conditions and is more comparable across dates.


select_bands(
    unzipped_dir=unzipped_dir,
    band_suffixes=["B11_20m.jp2", "B08_10m.jp2"], 
    # B11 refers to the Short-Wave Infrared (SWIR) band centered around 1610 nm with a 20 m resolution.
    # B08 refers to the Near-Infrared (NIR) band centered around 842 nm with a 10 m resolution.
    target_crs="EPSG:4326",
    resample=10,
    coregister=True,
    AOI= None, #aoi_polygon,
    output_dir="./processed_data/selected_bands", 
    calibrate=True
)


In [ ]:
ls ./processed_data/selected_bands/
What Is NDBI?
NDBI stands for Normalized Difference Built-up Index. It is used to highlight built-up and urban areas by comparing reflectance in the SWIR and NIR bands.

The formula is:

$$ \mathrm{NDBI} = \frac{B11 - B08}{B11 + B08} $$

Bands Used
B11 = Band 11, also called SWIR (Short-Wave Infrared), centered at about 1610 nm, with 20 m spatial resolution.
B08 = Band 8, also called NIR (Near-Infrared), centered at about 842 nm, with 10 m spatial resolution.
Why These Bands?
SWIR is sensitive to built surfaces, soil dryness, and moisture differences.
NIR responds strongly to healthy vegetation.
Built-up areas often appear stronger in SWIR than in NIR, which makes NDBI useful for urban mapping and damage analysis.


input_dir = "./processed_data/selected_bands"
output_dir = "./processed_data/indices"
compute_ndbi(input_dir, output_dir)
# NDBI (Normalized Difference Built-up Index)
# is computed using the formula: NDBI = (B11 - B08) / (B11 + B08)
Exercice
Question:
Why is coregistration important for change detection?
Hint: Think about how misaligned images can affect pixel-by-pixel comparisons.

FAQ: Load Data, Coregistration, and Preprocessing
Q1: Why is coregistration important for change detection?

A: Coregistration ensures that images from different dates or sensors align spatially. Misaligned images can lead to incorrect pixel-by-pixel comparisons, resulting in inaccurate change detection.
Q2: What tools can be used for coregistration?

A: Python libraries like rasterio, geemap, and pyproj are commonly used for coregistration. These tools allow you to reproject and align images to a common coordinate reference system.
Q3: What preprocessing steps are necessary for Sentinel data?

A: Preprocessing typically includes cropping to the Area of Interest (AOI), reprojection to a common CRS (e.g., EPSG:4326), and resampling to a consistent resolution (e.g., 10 meters).
Q4: Can Sentinel-1 and Sentinel-2 data be used together?

A: Yes, Sentinel-1 SAR data and Sentinel-2 optical data can be combined for comprehensive analysis. Coregistration is essential to ensure spatial alignment between the two datasets.
Q5: What format should the processed data be saved in?

A: Processed data is typically saved in GeoTIFF format (.tif) for compatibility with GIS software and Python libraries.
5. Constructing and Visualizing Time Series
Theory
Building time series stacks involves organizing satellite images over time for a specific Area of Interest (AOI). Temporal resolution refers to the frequency at which images are captured, which is critical for detecting changes or trends.

Practice
Let's visualize a simple NDBI (Normalized Difference Built-up Index) time series for a conflict-affected Area of Interest (AOI). NDBI is a common index derived from satellite data (e.g., Sentinel-2) that indicates built-up area and can reveal abrupt changes due to development.
NDBI = (SWIR - NIR) / (SWIR + NIR)

We will construct and visualize a time series using real NDBI data files stored in the ./processed_data/indices directory. Using xarray, we can efficiently handle multi-dimensional raster data and perform operations such as aggregation, interpolation, and visualization.



In [ ]:
import xarray as xr
import rioxarray
import glob
import os
from datetime import datetime
import matplotlib.pyplot as plt

# Define the directory containing NDBI files
ndbi_dir = "./processed_data/indices"

# List all NDBI files
ndbi_files = sorted([f for f in glob.glob(os.path.join(ndbi_dir, "*.tif"))])

# Helper function to extract datetime from filenames
def extract_dates_from_filenames(filenames):
    return [datetime.strptime(f.split("_")[4][:8], "%Y%m%d") for f in filenames]

# Create a time dimension variable
time_var = xr.Variable("time", extract_dates_from_filenames(ndbi_files))

# Load all NDBI files into an xarray DataArray
ndbi_da = xr.concat([rioxarray.open_rasterio(f, masked=True).squeeze() for f in ndbi_files], dim=time_var)

# Add metadata attributes
ndbi_da.attrs["long_name"] = "Normalized Difference Built-up Index (NDBI)"
ndbi_da.attrs["units"] = "dimensionless"

# Display the DataArray
ndbi_da


def plot_mean_ndbi_time_series(ndbi_da, aoi_coords):
    aoi_coords_lonlat = [(lon, lat) for lat, lon in aoi_coords]
    aoi_polygon = Polygon(aoi_coords_lonlat)

    ndbi_clipped = ndbi_da.rio.clip([aoi_polygon], ndbi_da.rio.crs, drop=False)
    mean_ndbi = ndbi_clipped.mean(dim=["x", "y"], skipna=True)

    # Sort by time to ensure proper connection order
    mean_ndbi = mean_ndbi.sortby("time")

    plt.figure(figsize=(10,4))
    plt.plot(mean_ndbi["time"].values, mean_ndbi.values, marker="o", linestyle="-", label="Mean NDBI")
    plt.title("Mean NDBI Time Series (AOI)")
    plt.xlabel("Date")
    plt.ylabel("Mean NDBI")
    plt.grid(True)
    plt.legend()
    plt.show()
plot_mean_ndbi_time_series(ndbi_da, aoi_polygon) #Tel Aviv AOI


In [ ]:
# Plot the mean NDBI time series for the entire data
aoi_polygon
mean_ndbi = ndbi_da.mean(dim=["x", "y"])
# Sort by time to ensure proper connection order
mean_ndbi = mean_ndbi.sortby("time")
mean_ndbi.plot(marker="o", figsize=(10, 4), label="Mean NDBI")
plt.title("Mean NDBI Time Series")
plt.xlabel("Date")
plt.ylabel("Mean NDBI")
plt.grid(True)
plt.legend()
plt.show()


# Visualize NDBI for a specific date
ndbi_da.sel(time="2025-06-18").plot(cmap="coolwarm", figsize=(8, 6), robust=True)
plt.title("NDBI for 2025-06-18")
plt.show()


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon

def plot_ndbi_for_dates(ndbi_da, aoi_coords, dates):
    aoi_coords_lonlat = [(lon, lat) for lat, lon in aoi_coords]
    aoi_polygon = Polygon(aoi_coords_lonlat)

    # Clip once, then plot each selected date in a grid
    ndbi_clipped = ndbi_da.rio.clip([aoi_polygon], ndbi_da.rio.crs, drop=True)
    ncols = min(3, len(dates))
    nrows = math.ceil(len(dates) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, date in zip(axes, dates):
        ndbi_slice = ndbi_clipped.sel(time=date)
        ndbi_slice.plot(ax=ax, cmap="coolwarm", robust=True, add_colorbar=False)
        ax.set_title(f"NDBI for {date} (AOI)")
        ax.set_xlabel("")
        ax.set_ylabel("")

    for ax in axes[len(dates):]:
        ax.axis("off")

    fig.tight_layout()
    plt.show()


dates = ["2025-06-03", "2025-06-18", "2025-06-20", "2025-06-23", "2025-06-28"]
plot_ndbi_for_dates(ndbi_da, aoi_polygon, dates)
Exercise
Identify the date with the highest mean NDBI value from the time series plot.


In [ ]:
import pandas as pd
from IPython.display import display

# Find the date with the highest mean NDBI value
mean_ndbi = ndbi_da.mean(dim=["x", "y"], skipna=True).sortby("time")

ranked_table = mean_ndbi.to_dataframe(name="mean_ndbi").reset_index()
ranked_table["time"] = pd.to_datetime(ranked_table["time"])
ranked_table["date"] = ranked_table["time"].dt.strftime("%d %B %Y")
ranked_table["year"] = ranked_table["time"].dt.year
ranked_table = ranked_table.sort_values("mean_ndbi", ascending=False).reset_index(drop=True)
ranked_table.insert(0, "rank", ranked_table.index + 1)

display(ranked_table[["rank", "date", "year", "mean_ndbi"]])

highest_row = ranked_table.iloc[0]
print(f"Highest mean NDBI date: {highest_row['date']}")
print(f"Highest mean NDBI value: {highest_row['mean_ndbi']:.4f}")
FAQ: Constructing and Visualizing Time Series
Q1: Can xarray handle large datasets efficiently?

A: Yes, xarray is optimized for handling multi-dimensional data and supports lazy loading, which allows processing large datasets without loading everything into memory.
Q2: What are the advantages of using xarray for time series analysis?

A: Xarray provides labeled dimensions and coordinates, making it easier to manipulate and analyze time series data. It also integrates well with other Python libraries like matplotlib for visualization.
6. Patch-Based NDBI Analysis
Theory
In this section, we split the AOI into regular patches and analyze patch-level NDBI time series. This helps localize where the strongest temporal transitions occur.

Practice
Start with n_patches=16 (4x4).
Use the first NDBI date as a raster background and draw the patch grid on top (xarray/matplotlib).
For each patch, plot NDBI across selected dates and highlight the strongest transition point.
Rank patches by transition score.
Refine iteratively by selecting a patch ID and re-running with n_patches=4 (2x2) on the selected patch AOI.


import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from shapely.geometry import Polygon, box, mapping

def latlon_to_polygon(aoi_coords_latlon):
    # Input format: [(lat, lon), ...]
    return Polygon([(lon, lat) for lat, lon in aoi_coords_latlon])

def polygon_to_latlon_coords(poly):
    # Output format: [(lat, lon), ...]
    return [(lat, lon) for lon, lat in list(poly.exterior.coords)]

def build_patch_grid(aoi_coords_latlon, n_patches=16):
    side = int(math.sqrt(n_patches))
    if side * side != n_patches:
        raise ValueError("n_patches must be a perfect square (e.g., 4, 9, 16, 25)")

    aoi_poly = latlon_to_polygon(aoi_coords_latlon)
    minx, miny, maxx, maxy = aoi_poly.bounds
    dx = (maxx - minx) / side
    dy = (maxy - miny) / side

    patches = []
    patch_id = 0
    for r in range(side):
        for c in range(side):
            x0 = minx + c * dx
            x1 = x0 + dx
            y0 = miny + r * dy
            y1 = y0 + dy
            cell = box(x0, y0, x1, y1)
            inter = aoi_poly.intersection(cell)
            if not inter.is_empty and inter.area > 0:
                if inter.geom_type != "Polygon":
                    inter = max(inter.geoms, key=lambda g: g.area)
                patches.append({"patch_id": patch_id, "polygon": inter})
            patch_id += 1

    return aoi_poly, patches

def plot_patch_grid_on_raster(ndbi_da, aoi_poly, patches, date=None):
    if date is None:
        raster_slice = ndbi_da.isel(time=0)
        date_label = pd.to_datetime(ndbi_da["time"].values[0]).strftime("%Y-%m-%d")
    else:
        raster_slice = ndbi_da.sel(time=date)
        date_label = str(date)

    # Crop to the selected AOI to keep rendering fast and focused
    try:
        raster_slice = raster_slice.rio.clip([mapping(aoi_poly)], ndbi_da.rio.crs, drop=True)
    except Exception:
        minx, miny, maxx, maxy = aoi_poly.bounds
        raster_slice = raster_slice.sel(x=slice(minx, maxx), y=slice(maxy, miny))

    fig, ax = plt.subplots(figsize=(8, 8))
    raster_slice.plot(ax=ax, cmap="coolwarm", robust=True)

    aoi_x, aoi_y = aoi_poly.exterior.xy
    ax.plot(aoi_x, aoi_y, color="blue", linewidth=2, label="AOI")

    for patch in patches:
        poly = patch["polygon"]
        pid = patch["patch_id"]
        x, y = poly.exterior.xy
        ax.plot(x, y, color="yellow", linewidth=1.0)
        c = poly.centroid
        ax.text(c.x, c.y, str(pid), fontsize=8, color="black", ha="center", va="center",
                bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=1.5))

    ax.set_title(f"AOI Patch Grid over NDBI ({date_label})")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend(loc="upper right")
    ax.grid(False)
    plt.tight_layout()
    plt.show()

def compute_patch_time_series(ndbi_da, patches, dates=None):
    da = ndbi_da
    if dates is not None:
        da = da.sel(time=dates)

    times = pd.to_datetime(da["time"].values)
    rows = []

    for patch in patches:
        pid = patch["patch_id"]
        poly = patch["polygon"]
        try:
            patch_da = da.rio.clip([mapping(poly)], da.rio.crs, drop=True)
            ts = patch_da.mean(dim=["x", "y"], skipna=True).values.astype(float)
        except Exception:
            ts = np.full(len(times), np.nan)

        row = {"patch_id": pid}
        for i, t in enumerate(times):
            row[t] = ts[i]
        rows.append(row)

    df = pd.DataFrame(rows).sort_values("patch_id").reset_index(drop=True)
    return df, times

def detect_transition_point(series, method="max_abs_diff"):
    y = np.asarray(series, dtype=float)
    if len(y) < 2 or np.all(np.isnan(y)):
        return 0, np.nan

    y_clean = np.copy(y)
    nan_mask = np.isnan(y_clean)
    if np.any(nan_mask):
        valid = np.where(~nan_mask)[0]
        if len(valid) == 0:
            return 0, np.nan
        y_clean[nan_mask] = np.interp(np.where(nan_mask)[0], valid, y_clean[valid])

    if method == "max_abs_gradient":
        score_series = np.abs(np.gradient(y_clean))
        idx = int(np.nanargmax(score_series))
        score = float(score_series[idx])
    else:
        diff = np.diff(y_clean)
        if method == "zscore_diff":
            s = np.std(diff)
            if s == 0:
                score_series = np.abs(diff)
            else:
                score_series = np.abs((diff - np.mean(diff)) / s)
        else:
            # Default and most suitable for abrupt transitions
            score_series = np.abs(diff)
        jump_idx = int(np.nanargmax(score_series))
        idx = min(jump_idx + 1, len(y_clean) - 1)
        score = float(score_series[jump_idx])

    return idx, score

def plot_patch_time_series_grid(patch_df, times, method="max_abs_diff", sort_by="score"):
    if sort_by not in {"score", "value"}:
        raise ValueError("sort_by must be 'score' or 'value'")

    n = len(patch_df)
    ncols = min(4, n)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.8 * nrows), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()

    time_cols = list(times)
    summary = []

    for _, row in patch_df.iterrows():
        pid = int(row["patch_id"])
        y = row[time_cols].values.astype(float)
        t_idx, score = detect_transition_point(y, method=method)
        summary.append({
            "patch_id": pid,
            "transition_time": pd.to_datetime(times[t_idx]).strftime("%Y-%m-%d"),
            "transition_value": y[t_idx],
            "transition_score": score
        })

    summary_df = pd.DataFrame(summary)
    summary_df = summary_df.sort_values(f"transition_{sort_by}", ascending=False).reset_index(drop=True)

    for i, (_, row) in enumerate(patch_df.iterrows()):
        ax = axes[i]
        pid = int(row["patch_id"])
        y = row[time_cols].values.astype(float)
        t_idx, score = detect_transition_point(y, method=method)
        transition_time = times[t_idx]
        transition_value = y[t_idx]

        ax.plot(times, y, marker="o", linewidth=1.6)
        ax.scatter(transition_time, transition_value, color="red", s=40, zorder=3, label="Highest score")

        value_idx = int(np.nanargmax(y)) if np.any(~np.isnan(y)) else 0
        ax.scatter(times[value_idx], y[value_idx], color="orange", s=40, zorder=3, label="Highest value")

        ax.set_title(f"Patch {pid}")
        ax.grid(True, alpha=0.3)

    for j in range(n, len(axes)):
        axes[j].axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper right")
    fig.suptitle(f"Patch-wise NDBI Time Series ({method}, sort_by={sort_by})", y=1.02)
    fig.tight_layout()
    plt.show()

    return summary_df

def run_patch_analysis(ndbi_da, aoi_coords_latlon, n_patches=16, dates=None, method="max_abs_diff", show_map=True, grid_date=None, sort_by="score"):
    aoi_poly, patches = build_patch_grid(aoi_coords_latlon, n_patches=n_patches)

    if show_map:
        plot_patch_grid_on_raster(ndbi_da, aoi_poly, patches, date=grid_date)

    patch_df, times = compute_patch_time_series(ndbi_da, patches, dates=dates)
    summary_df = plot_patch_time_series_grid(patch_df, times, method=method, sort_by=sort_by)

    best_patch_id = int(summary_df.iloc[0]["patch_id"])
    result = {
        "aoi_poly": aoi_poly,
        "patches": patches,
        "patches_latlon": [polygon_to_latlon_coords(p["polygon"]) for p in patches],
        "summary": summary_df,
        "best_patch_id": best_patch_id
    }
    return result
Workflow
Split the current AOI into patches and compute mean NDBI per patch over time.
Plot all patch time series in a grid and highlight the highest transition point in each subplot.
Rank patches by either transition value or transition score.
Select the patch you want (or any patch by ID), treat it as a new AOI, and split again.
Repeat iteratively (Level 1 -> Level 2 -> Level 3).
Score vs Value

Transition score: how strong the change is at the detected transition point.
Transition value: the actual NDBI value at that transition point.
A patch can have a high score but a low value, or a high value but a small score.
Use sort_by="score" when you want the strongest change, and sort_by="value" when you care about the highest NDBI level at the transition point.
Score formula

For the default method max_abs_diff, the score is:

$$ score_t = |p_t - p_{t-1}| $$

where:

$p_t$ is the mean NDBI value at time $t$ for the patch.
$p_{t-1}$ is the mean NDBI value at the previous time step.
For the other optional methods:

$$ score_t = |\nabla p_t| \quad \text{for } max_abs_gradient $$

$$ score_t = \left|\frac{p_t - \mu_{\Delta p}}{\sigma_{\Delta p}}\right| \quad \text{for } zscore_diff $$

Transition methods (selectable):

max_abs_diff (default): max of $|p_t - p_{t-1}|$
max_abs_gradient: max of $|\nabla p_t|$
zscore_diff: max standardized difference across steps


In [ ]:
aoi_polygon = [
    [32.15, 34.70],  # NW corner
    [32.15, 34.85],  # NE corner
    [32.05, 34.85],  # SE corner
    [32.05, 34.70],  # SW corner
    [32.15, 34.70]   # Closing the polygon
]


# Level 1: split the AOI into 16 patches (4x4) and analyze transitions
analysis_dates = ["2025-06-03", "2025-06-18", "2025-06-20", "2025-06-23", "2025-06-28"]
level1_result = run_patch_analysis(
    ndbi_da=ndbi_da,
    aoi_coords_latlon=aoi_polygon,
    n_patches=16,
    dates=analysis_dates,
    method="max_abs_diff",  # options: max_abs_diff, max_abs_gradient, zscore_diff
    sort_by="score",  # options: score, value
    show_map=True,
    grid_date=analysis_dates[0]
)

display(level1_result["summary"])
print(f"Highest transition patch at Level 1: {level1_result['best_patch_id']}")


In [ ]:
# Level 2: pick one patch from Level 1 and split again into 4 subpatches
selected_patch_id_lvl1 = 11  # Change this to any patch ID from level1_result["summary"]
selected_patch_aoi = level1_result["patches_latlon"][selected_patch_id_lvl1]

level2_result = run_patch_analysis(
    ndbi_da=ndbi_da,
    aoi_coords_latlon=selected_patch_aoi,
    n_patches=4,
    dates=analysis_dates,
    method="max_abs_diff",
    sort_by="score",
    show_map=True,
    grid_date=analysis_dates[0]
 )

display(level2_result["summary"])
print(f"Highest transition patch at Level 2: {level2_result['best_patch_id']}")


# Level 3: refine again by selecting one patch from Level 2
selected_patch_id_lvl2 = 3  # Change this to any patch ID from level2_result["summary"]
selected_patch_aoi_lvl2 = level2_result["patches_latlon"][selected_patch_id_lvl2]

level3_result = run_patch_analysis(
    ndbi_da=ndbi_da,
    aoi_coords_latlon=selected_patch_aoi_lvl2,
    n_patches=4,
    dates=analysis_dates,
    method="max_abs_diff",
    sort_by="score",
    show_map=True,
    grid_date=analysis_dates[0]
 )

display(level3_result["summary"])
print(f"Highest transition patch at Level 3: {level3_result['best_patch_id']}")
7. Change Detection Algorithms
Theory
Change detection algorithms are used to identify differences between images captured at different times. These differences can be caused by natural events, human activities, or other phenomena. Common techniques include:

Image differencing: Subtracting one image from another to highlight changes.
Thresholding: Applying a threshold to the differenced image to classify areas of change.
Time series break detection: Identifying abrupt changes in time series data using statistical methods (e.g., BFAST, Ruptures).
These methods are particularly useful for monitoring conflict zones, urban development, or environmental changes.

Practice
We will implement image differencing and thresholding to detect changes between pre-strike and post-strike images. This example uses Sentinel-2 NDBI data to identify urban damage caused by missile strikes.



In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
def plot_ndbi_difference(ndbi_da, aoi_coords, pre_date, post_date, threshold=-0.1):
    aoi_coords_lonlat = [(lon, lat) for lat, lon in aoi_coords]
    aoi_polygon = Polygon(aoi_coords_lonlat)

    ndbi_clipped = ndbi_da.rio.clip([aoi_polygon], ndbi_da.rio.crs, drop=True)

    pre_ndbi = ndbi_clipped.sel(time=pre_date)
    post_ndbi = ndbi_clipped.sel(time=post_date)

    diff = post_ndbi - pre_ndbi
    change_mask = diff < threshold

    plt.figure(figsize=(8,6))
    diff.plot(cmap="coolwarm", robust=True)
    plt.title(f"NDBI Difference ({post_date} - {pre_date})")
    plt.show()

    plt.figure(figsize=(8,6))
    change_mask.plot(cmap="gray")
    plt.title(f"Change Mask (Threshold: {threshold})")
    plt.show()
    
   
# Load pre-strike and post-strike NDBI images
pre_strike_date = "2025-06-03"
post_strike_date = "2025-06-28"


post_date = post_strike_date
plot_ndbi_difference(ndbi_da,
                     aoi_polygon,
                     pre_strike_date,
                     post_strike_date,
                     threshold=-0.2)


import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

def plot_ndbi_difference_with_mask(ndbi_da, aoi_coords, pre_date, post_date, threshold=-0.1, alpha=1.0):
    """
    Plot NDBI difference with a transparent mask overlay where True values remain transparent.

    Parameters:
        ndbi_da (xarray.DataArray): NDBI data array.
        aoi_coords (list): Coordinates of the AOI polygon.
        pre_date (str): Date of the pre-strike image.
        post_date (str): Date of the post-strike image.
        threshold (float): Threshold for change detection.
        alpha (float): Transparency factor for the mask (0 to 1).
    """
    aoi_coords_lonlat = [(lon, lat) for lat, lon in aoi_coords]
    aoi_polygon = Polygon(aoi_coords_lonlat)

    # Clip NDBI data to AOI
    ndbi_clipped = ndbi_da.rio.clip([aoi_polygon], ndbi_da.rio.crs, drop=True)

    # Select pre-strike and post-strike NDBI images
    pre_ndbi = ndbi_clipped.sel(time=pre_date)
    post_ndbi = ndbi_clipped.sel(time=post_date)

    # Compute NDBI difference
    diff = post_ndbi - pre_ndbi

    # Create change mask based on threshold
    change_mask = diff < threshold

    # Convert mask to RGBA format
    rgba_mask = np.zeros((*change_mask.shape, 4), dtype=np.float32)  # Initialize RGBA array
    rgba_mask[..., 0:3] = 0  # Set RGB channels to black
    rgba_mask[..., 3] = np.where(change_mask, 0, alpha)  # Set alpha channel (0 for True, alpha for False)

    # Plot NDBI difference with transparent mask overlay
    plt.figure(figsize=(8, 6))
    plt.imshow(diff, cmap="coolwarm", origin="upper", alpha=1.0)  # Original NDBI difference
    plt.imshow(rgba_mask, origin="upper")  # Transparent mask overlay
    plt.title(f"NDBI Difference with Transparent Mask ({post_date} - {pre_date})")
    plt.colorbar(label="NDBI Difference")
    plt.show()

# Example usage
pre_strike_date = "2025-06-03"
post_strike_date = "2025-06-28"
plot_ndbi_difference_with_mask(ndbi_da, aoi_polygon, pre_strike_date, post_strike_date, threshold=-0.2, alpha=0.7)
Exercise
Adjust the threshold value and observe how the change mask varies.
FAQ: Change Detection Algorithms
Q1: Why is image differencing useful for change detection?

A: Image differencing highlights areas of change by subtracting one image from another. This is particularly effective for detecting abrupt changes caused by events like missile strikes or natural disasters.
Q2: What is the purpose of thresholding in change detection?

A: Thresholding classifies areas of change based on the magnitude of the difference. It helps isolate significant changes from minor variations.
Q3: What factors should be considered when selecting a threshold value?

A: The threshold value should be chosen based on the expected magnitude of change and the characteristics of the data. It may require tuning to balance sensitivity and specificity.
8. Patched Change Detection
Theory
After identifying the most interesting patch from the patch-based analysis, we repeat change detection only inside that selected patch AOI. This keeps the comparison local and makes the change maps easier to interpret.

Practice
We will reuse the same change-detection plots, but now only for selected_patch_aoi_lvl2.



In [ ]:
# Patched change detection inside the selected Level-2 patch AOI
patched_aoi = selected_patch_aoi_lvl2
patched_pre_date = "2025-06-03"
patched_post_date = "2025-06-28"

plot_ndbi_difference(
    ndbi_da,
    patched_aoi,
    patched_pre_date,
    patched_post_date,
    threshold=-0.2,
 )

plot_ndbi_difference_with_mask(
    ndbi_da,
    patched_aoi,
    patched_pre_date,
    patched_post_date,
    threshold=-0.2,
    alpha=0.7,
 )
Further Improvements for Your Analysis
1. Radiometric Calibration
Apply radiometric calibration to Sentinel-2 bands (B11 and B08) to ensure consistent intensity values.
Use tools like Sen2Cor for atmospheric correction.
Example: Adjust gain and offset values for calibration.
2. Cloud and Shadow Masking
Filter out cloudy pixels using Sentinel-2 cloud masks (QA60 or SCL bands).
Use machine learning-based tools like s2cloudless for advanced cloud masking.
Example: Mask clouds and shadows to reduce noise.
3. Noise Reduction
Apply smoothing techniques like Gaussian filters or median filters to reduce pixel-level noise.
Example: Use scipy.ndimage.gaussian_filter for noise reduction.
4. Topographic Correction
Correct for terrain-induced illumination effects using tools like ATCOR.
Example: Implement slope and aspect-based correction formulas.
5. Validation
Validate NDBI results against reference data or ground truth.
Use metrics like RMSE or visual inspection to ensure accuracy.
6. Improved Visualization
Overlay cloud masks or highlight areas with significant changes for better interpretation.
Example: Use matplotlib or folium for enhanced visualizations.
7. Temporal Analysis
Perform time series break detection using statistical methods like BFAST or Ruptures.
Example: Detect abrupt changes in NDBI values over time.
8. Integration of Sentinel-1 SAR Data
Combine Sentinel-1 SAR data with Sentinel-2 optical data for comprehensive analysis.
Example: Use SAR backscatter changes to complement NDBI results.
Useful Links
Sen2Cor Atmospheric Correction
s2cloudless Cloud Masking
ATCOR Topographic Correction
BFAST Time Series Analysis
Ruptures Change Point Detection
Rasterio Python Library
